In [ ]:
# ===== plan_d_c — CELL 0 : setup =====
# কাজ একটাই: text+text আর image+image জোড়ার জন্য zero-shot cross-encoder score বানানো।
# image+text আগেই হয়ে গেছে (rr_train.npy / rr_test.npy, effect 2.13)।
# এখানে training নেই — সব কিছু plan_d_final নেবে।
#
# Accelerator: GPU (T4 x2 বা P100)।  Internet: ON।
# Input: essentials, img384, ocr_384x2
import os, glob, json, time, gc, sys, subprocess, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

OUT, INP = '/kaggle/working', '/kaggle/input'
T0 = time.time()
def find(n, isdir=False):
    for r in (INP, OUT):
        for h in glob.glob(f'{r}/**/{n}', recursive=True):
            if os.path.isdir(h) == isdir: return h
    return None
def tlog(*a): print(f'[{(time.time()-T0)/60:6.1f} min]', *a, flush=True)

import torch
tlog('VRAM', f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB',
     '| GPUs', torch.cuda.device_count())

ESS  = os.path.dirname(find('meta_train.parquet'))
IMGD = find('img384', isdir=True)
mtr = pd.read_parquet(f'{ESS}/meta_train.parquet')
mte = pd.read_parquet(f'{ESS}/meta_test.parquet')
for d in (mtr, mte):
    for c in ['t1','t2','h1','h2']: d[c] = d[c].astype(str)
    d['combo'] = np.where(d.t1 < d.t2, d.t1+'+'+d.t2, d.t2+'+'+d.t1)
mtr['y'] = mtr['label'].astype(str)

texts = pd.read_parquet(f'{ESS}/texts.parquet'); texts['hash'] = texts['hash'].astype(str)
TXTMAP = dict(zip(texts.hash, texts.text.fillna('').astype(str)))

p = find('ocr_384x2.parquet') or find('ocr.parquet')
OCRMAP = {}
if p:
    o = pd.read_parquet(p)
    OCRMAP = dict(zip(o.hash.astype(str), o.ocr.fillna('').astype(str)))
    tlog('OCR:', p, '| entries', len(OCRMAP))
else:
    print('OCR নেই — image+image-এ শুধু ছবি যাবে')

def subset(m, cb):
    return m[m.combo == cb].reset_index(drop=True)

tlog('train', mtr.shape, 'test', mte.shape)
for cb in ['image+text','text+text','image+image']:
    tlog(f'  {cb}: train {len(subset(mtr,cb))}  test {len(subset(mte,cb))}')


In [ ]:
# ===== plan_d_c — CELL 1 : reranker চালানোর সাধারণ যন্ত্র =====
# torchao 0.10.0 সরানো — না হলে transformers import-এ আটকে দেয় (নিচে ব্যাখ্যা)
subprocess.run(f'{sys.executable} -m pip uninstall -y -q torchao', shell=True, timeout=900)
subprocess.run(f'{sys.executable} -m pip install -q -U "sentence-transformers>=5" qwen-vl-utils',
               shell=True, timeout=1800)
from sentence_transformers import CrossEncoder

BUDGET = 200 * 60          # প্রতি subset-এ সর্বোচ্চ সময়

def load_ce(name):
    tlog('loading', name)
    m = CrossEncoder(name, device='cuda', model_kwargs={'torch_dtype': torch.float16})
    tlog('loaded', name)
    return m

def gate(model, items_a, items_b, labels, instr, n=300, bs=8):
    # ৩০০ জোড়ায় class-ভিত্তিক গড় — effect 0.8 না ছাড়ালে ওই subset বাদ।
    idx = np.random.default_rng(0).choice(len(labels), min(n, len(labels)), replace=False)
    t = time.time()
    s = np.asarray(model.predict([(items_a[i], items_b[i]) for i in idx], prompt=instr,
                                 batch_size=bs, show_progress_bar=False,
                                 activation_fn=torch.nn.Sigmoid()), dtype=np.float32).reshape(-1)
    per = (time.time()-t)/len(idx)
    g = pd.Series(s).groupby(np.asarray(labels)[idx]).mean()
    eff = float((g.max()-g.min())/max(s.std(), 1e-9))
    print(g.round(4).to_string())
    print(f'ব্যাপ্তি {g.max()-g.min():.4f} | effect {eff:.2f} | {per:.2f}s/জোড়া')
    print(f'মাপকাঠি: image+text-এ effect ছিল 2.13। effect < 0.8 হলে এই subset বাদ দাও।')
    return eff, per

def score_all(model, tag, items_a, items_b, instr, bs=8, chunk=256, both_orders=True):
    # দুই ক্রমে score করে গড় — জোড়া অসমমিত নয়, তাই এটা noise কমায়।
    ck = f'{OUT}/_ck_{tag}.npy'
    done = list(np.load(ck)) if os.path.exists(ck) else []
    if done: tlog(f'{tag}: checkpoint থেকে {len(done)}')
    t = time.time()
    for i in range(len(done), len(items_a), chunk):
        pa, pb = items_a[i:i+chunk], items_b[i:i+chunk]
        s = np.asarray(model.predict(list(zip(pa, pb)), prompt=instr, batch_size=bs,
                       show_progress_bar=False, activation_fn=torch.nn.Sigmoid()),
                       dtype=np.float32).reshape(-1)
        if both_orders:
            s2 = np.asarray(model.predict(list(zip(pb, pa)), prompt=instr, batch_size=bs,
                            show_progress_bar=False, activation_fn=torch.nn.Sigmoid()),
                            dtype=np.float32).reshape(-1)
            s = (s + s2) / 2
        done.extend(s)
        np.save(ck, np.array(done, dtype=np.float32))
        el = time.time()-t; nd = max(len(done)-1, 1)
        tlog(f'  {tag} {len(done)}/{len(items_a)}  বাকি ~{(len(items_a)-len(done))*el/nd/60:.0f}m')
        if time.time()-t > BUDGET:
            tlog(f'সময়সীমা — {tag} {len(done)}/{len(items_a)} পর্যন্ত'); break
    a = np.array(done, dtype=np.float32)
    if len(a) < len(items_a):
        # অসম্পূর্ণ হলে বাকিটা গড় — কিন্তু তখন ফাইলটা ব্যবহার কোরো না, নিচের যাচাই ধরিয়ে দেবে
        a = np.concatenate([a, np.full(len(items_a)-len(a), float(a.mean()) if len(a) else 0.5, np.float32)])
    return a, len(done) >= len(items_a)

REPORT = {}
def finish(tag, cb, str_tr, str_te, ok_tr, ok_te, eff):
    np.save(f'{OUT}/rr_train_{tag}.npy', str_tr)
    np.save(f'{OUT}/rr_test_{tag}.npy',  str_te)
    g = pd.Series(str_tr).groupby(subset(mtr, cb).y.values).mean()
    e2 = float((g.max()-g.min())/max(str_tr.std(), 1e-9))
    REPORT[cb] = {'gate_effect': round(eff,3), 'full_effect': round(e2,3),
                  'complete_train': bool(ok_tr), 'complete_test': bool(ok_te),
                  'by_class': {k: round(float(v),4) for k, v in g.items()}}
    print(f'\n=== {cb} পূর্ণ train score ===')
    print(g.round(4).to_string()); print(f'effect {e2:.2f}')
    if not (ok_tr and ok_te):
        print('⚠️ অসম্পূর্ণ — plan_d_final-এ এই ফাইল ব্যবহার কোরো না')
    tlog(f'saved rr_train_{tag}.npy / rr_test_{tag}.npy')


In [ ]:
# ===== plan_d_c — CELL 2 : image+image (আগে — AUC সবচেয়ে খারাপ 0.672 / 0.683) =====
# দুটো figure একই paper-এর কিনা। ছবির সাথে OCR জুড়ে দিচ্ছি, কারণ কোন telescope /
# catalog ID / object name ফিরে এসেছে সেটাই same_paper-এর আসল প্রমাণ।
MODEL_VL = 'Qwen/Qwen3-VL-Reranker-2B'
INSTR_II = ('Given a figure from a scientific paper, retrieve another figure that comes '
            'from the same paper.')

model = load_ce(MODEL_VL)

# image+text-এ query ছিল সাদা text, document ছিল {'image': ...}। এখানে দুই পাশেই ছবি —
# query অবস্থানে dict চলে কিনা model card থেকে নিশ্চিত নয়। তাই তিনটে রূপ পরীক্ষা করে
# প্রথম যেটা চলে সেটাই নিই। ৪টা জোড়ায়, ১০ সেকেন্ড — Kaggle-এ ঘণ্টা নষ্ট হওয়ার চেয়ে ভালো।
FORMS = [
    ('image+ocr dict', lambda h: ({'image': f'{IMGD}/{h}.jpg', 'text': OCRMAP.get(h,'')[:600]}
                                  if OCRMAP.get(h) else {'image': f'{IMGD}/{h}.jpg'})),
    ('image dict',     lambda h: {'image': f'{IMGD}/{h}.jpg'}),
    ('ocr text',       lambda h: (OCRMAP.get(h,'')[:600] or 'a scientific figure')),
]
dii_tr, dii_te = subset(mtr, 'image+image'), subset(mte, 'image+image')
img_item = None
for nm, fn in FORMS:
    try:
        pr = [(fn(a), fn(b)) for a, b in zip(dii_tr.h1[:4], dii_tr.h2[:4])]
        v = model.predict(pr, prompt=INSTR_II, batch_size=2, show_progress_bar=False,
                          activation_fn=torch.nn.Sigmoid())
        v = np.asarray(v, dtype=np.float32).reshape(-1)
        assert len(v) == 4 and np.isfinite(v).all() and v.std() > 0
        img_item = fn; print(f'✅ input রূপ: {nm} | নমুনা {np.round(v,3).tolist()}'); break
    except Exception as e:
        print(f'✗ {nm}: {repr(e)[:160]}')
if img_item is None:
    raise RuntimeError('কোনো রূপেই image+image চলল না — CELL 3 (text+text) এ চলে যাও')

A_tr = [img_item(h) for h in dii_tr.h1]; B_tr = [img_item(h) for h in dii_tr.h2]
A_te = [img_item(h) for h in dii_te.h1]; B_te = [img_item(h) for h in dii_te.h2]
print('\n=== GATE : image+image ===')
eff_ii, per = gate(model, A_tr, B_tr, dii_tr.y.values, INSTR_II, bs=4)
print(f'৬০০০ জোড়া × ২ ক্রম ≈ {per*12000/60:.0f} মিনিট')

if eff_ii >= 0.8:
    s_tr, ok1 = score_all(model, 'ii_tr', A_tr, B_tr, INSTR_II, bs=4)
    s_te, ok2 = score_all(model, 'ii_te', A_te, B_te, INSTR_II, bs=4)
    finish('image_image', 'image+image', s_tr, s_te, ok1, ok2, eff_ii)
else:
    print('effect খুব কম — image+image বাদ, সময় বাঁচাও')
    REPORT['image+image'] = {'gate_effect': round(eff_ii,3), 'skipped': True}


In [ ]:
# ===== plan_d_c — CELL 3 : text+text =====
# এখানে ছবি নেই, তাই text-only reranker ভালো হওয়ার কথা — সস্তাও।
# লোড না হলে VL reranker-এই ফিরে যাই।
MODEL_TXT = 'Qwen/Qwen3-Reranker-4B'
INSTR_TT = ('Given a figure caption from a scientific paper, retrieve another caption '
            'that comes from the same paper.')

del model; gc.collect(); torch.cuda.empty_cache()
try:
    model = load_ce(MODEL_TXT)
except Exception as e:
    print('text reranker ব্যর্থ:', repr(e)[:300], '→ VL reranker দিয়েই চালাচ্ছি')
    model = load_ce(MODEL_VL)

dtt_tr, dtt_te = subset(mtr, 'text+text'), subset(mte, 'text+text')
cut = lambda h: TXTMAP.get(h, '')[:1800]
A_tr = [cut(h) for h in dtt_tr.h1]; B_tr = [cut(h) for h in dtt_tr.h2]
A_te = [cut(h) for h in dtt_te.h1]; B_te = [cut(h) for h in dtt_te.h2]

print('\n=== GATE : text+text ===')
eff_tt, per = gate(model, A_tr, B_tr, dtt_tr.y.values, INSTR_TT, bs=16)
print(f'৬০০০ জোড়া × ২ ক্রম ≈ {per*12000/60:.0f} মিনিট')

if eff_tt >= 0.8:
    s_tr, ok1 = score_all(model, 'tt_tr', A_tr, B_tr, INSTR_TT, bs=16)
    s_te, ok2 = score_all(model, 'tt_te', A_te, B_te, INSTR_TT, bs=16)
    finish('text_text', 'text+text', s_tr, s_te, ok1, ok2, eff_tt)
else:
    print('effect খুব কম — text+text বাদ')
    REPORT['text+text'] = {'gate_effect': round(eff_tt,3), 'skipped': True}


In [ ]:
# ===== plan_d_c — CELL 4 : boundary AUC — এটাই আসল রায় =====
# gate effect বলে class আলাদা হচ্ছে কিনা। কিন্তু আমাদের দরকার ঠিক দুটো সীমানা:
#   same_paper vs related   (এখন image+image-এ AUC 0.672)
#   related vs unrelated    (এখন image+image-এ AUC 0.683)
# নতুন score এই দুটোয় না বাড়ালে সেটা plan_d_final-এ কিছু দেবে না।
from sklearn.metrics import roc_auc_score
BASE = {'image+image': (0.672, 0.683), 'text+text': (0.797, 0.846)}

for cb, tag in [('image+image','image_image'), ('text+text','text_text')]:
    p = f'{OUT}/rr_train_{tag}.npy'
    if not os.path.exists(p): continue
    s = np.load(p); y = subset(mtr, cb).y.values
    print(f'\n{cb}  (একা reranker score, বনাম এখনকার পুরো pipeline)')
    for i, (a, b) in enumerate([('same_paper','related_papers'), ('related_papers','unrelated_papers')]):
        m = np.isin(y, [a, b]); t = (y[m] == a).astype(int)
        auc = roc_auc_score(t, s[m])
        auc = max(auc, 1-auc)
        print(f'  {a:15s} vs {b:17s}  reranker {auc:.3f}   pipeline {BASE[cb][i]:.3f}')

json.dump(REPORT, open(f'{OUT}/report_C.json','w'), indent=1, default=str)
print('\n' + json.dumps(REPORT, indent=1, default=str))
print('\n👉 Save Version → Output কে Dataset বানাও → plan_d_final-এ attach করো।')
tlog('done')
